# Love as a Two-Agent Control System

This notebook models a relationship as a **two-agent feedback control system** with:

- self-decay / emotional forgetting
- partner response gains
- reward-based reinforcement
- setpoint tracking
- delay
- noise
- interactive sliders

The model is intentionally stylized. It is useful for intuition and experimentation, not as a literal scientific theory of human relationships.


## Model

For emotional states \(x_1(t)\) and \(x_2(t)\):

\[
\dot{x}_1 =
-a_1 x_1
+ b_1 \tanh(x_2(t-\tau_1))
+ k_1 \tanh(x_2(t-\tau_1))
+ K_1(x_2(t-\tau_1)-x_1^*)
+ \text{shock}_1
+ \epsilon_1
\]

\[
\dot{x}_2 =
-a_2 x_2
+ b_2 \tanh(x_1(t-\tau_2))
+ k_2 \tanh(x_1(t-\tau_2))
+ K_2(x_1(t-\tau_2)-x_2^*)
+ \text{shock}_2
+ \epsilon_2
\]

### Parameter meanings

- \(a_i\): emotional decay / how quickly feelings fade without reinforcement
- \(b_i\): responsiveness to the partner
- \(k_i\): reward sensitivity to the partner's affection
- \(K_i\): regulation / repair effort toward the relationship setpoint
- \(x_i^*\): desired affection level or emotional setpoint
- \(\tau_i\): delay in response
- \(\sigma_i\): noise / ambiguity / stress

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button
from dataclasses import dataclass

plt.rcParams["figure.figsize"] = (12, 8)

In [ ]:
@dataclass
class LoveSystemParams:
    T: float = 80.0
    dt: float = 0.05
    seed: int = 7

    x1_0: float = 0.2
    x2_0: float = -0.1

    a1: float = 0.8
    a2: float = 0.7

    b1: float = 1.4
    b2: float = 1.2

    k1: float = 0.9
    k2: float = 1.0

    x1_star: float = 0.6
    x2_star: float = 0.4

    K1: float = 0.5
    K2: float = 0.4

    tau1: float = 1.5
    tau2: float = 2.0

    sigma1: float = 0.04
    sigma2: float = 0.04

    sat_alpha: float = 1.0

    shock_time_1: float = 20.0
    shock_mag_1: float = 0.4
    shock_time_2: float = 45.0
    shock_mag_2: float = -0.5


def saturation(x, alpha=1.0):
    return np.tanh(alpha * x)


def external_shock(t, shock_time, shock_mag, width=0.8):
    return shock_mag * np.exp(-0.5 * ((t - shock_time) / width) ** 2)


def simulate(params: LoveSystemParams):
    rng = np.random.default_rng(params.seed)

    n_steps = int(params.T / params.dt) + 1
    t = np.linspace(0.0, params.T, n_steps)

    x1 = np.zeros(n_steps)
    x2 = np.zeros(n_steps)
    r1 = np.zeros(n_steps)
    r2 = np.zeros(n_steps)
    u1 = np.zeros(n_steps)
    u2 = np.zeros(n_steps)

    x1[0] = params.x1_0
    x2[0] = params.x2_0

    d1 = int(params.tau1 / params.dt)
    d2 = int(params.tau2 / params.dt)

    for i in range(1, n_steps):
        x2_delayed = x2[max(0, i - d1)]
        x1_delayed = x1[max(0, i - d2)]

        r1[i] = params.k1 * saturation(x2_delayed, params.sat_alpha)
        r2[i] = params.k2 * saturation(x1_delayed, params.sat_alpha)

        u1[i] = params.K1 * (x2_delayed - params.x1_star)
        u2[i] = params.K2 * (x1_delayed - params.x2_star)

        shock1 = external_shock(t[i], params.shock_time_1, params.shock_mag_1)
        shock2 = external_shock(t[i], params.shock_time_2, params.shock_mag_2)

        noise1 = params.sigma1 * rng.normal()
        noise2 = params.sigma2 * rng.normal()

        dx1 = (
            -params.a1 * x1[i - 1]
            + params.b1 * saturation(x2_delayed, params.sat_alpha)
            + r1[i]
            + u1[i]
            + shock1
            + noise1
        )

        dx2 = (
            -params.a2 * x2[i - 1]
            + params.b2 * saturation(x1_delayed, params.sat_alpha)
            + r2[i]
            + u2[i]
            + shock2
            + noise2
        )

        x1[i] = x1[i - 1] + params.dt * dx1
        x2[i] = x2[i - 1] + params.dt * dx2

    return t, x1, x2, r1, r2, u1, u2


def classify_regime(x1, x2):
    max_abs = max(np.max(np.abs(x1)), np.max(np.abs(x2)))
    if max_abs > 10:
        return "Divergent / unstable"

    tail1 = x1[-200:]
    tail2 = x2[-200:]
    amp1 = np.max(tail1) - np.min(tail1)
    amp2 = np.max(tail2) - np.min(tail2)
    slope1 = np.abs(np.mean(np.diff(tail1)))
    slope2 = np.abs(np.mean(np.diff(tail2)))

    if amp1 < 0.15 and amp2 < 0.15 and slope1 < 0.01 and slope2 < 0.01:
        return "Convergent / stable"
    return "Oscillatory / cycling"

## Two balls in a bowl

The simulation is also shown as **two marbles on a shared parabolic bowl** \(z \propto x^2 + y^2\). Each ball’s **azimuth** follows mainly its own emotional state; its **radius** on the bowl is pulled by the **partner’s** state, so the pair **co-orbits** on the surface. Stable runs drift toward the center (low “energy”); oscillatory regimes trace loops in plan view.

For standalone bowl animations, run `simulate` first (e.g. in a scratch cell) so `t`, `x1`, and `x2` exist, or use `interactive_dashboard()` for sliders plus bowl. The `animate_bowl_two_balls` helper returns HTML in Jupyter when available.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 — registers 3d projection
from matplotlib import animation as mpl_animation


def bowl_surface_mesh(r_max: float = 2.8, n: int = 55, k_parabola: float = 0.18):
    """Paraboloid z = k * (x^2 + y^2), clipped to a circular rim."""
    u = np.linspace(-r_max, r_max, n)
    v = np.linspace(-r_max, r_max, n)
    X, Y = np.meshgrid(u, v)
    R2 = X**2 + Y**2
    mask = R2 <= r_max**2
    Z = np.where(mask, k_parabola * R2, np.nan)
    return X, Y, Z, k_parabola


def coupled_bowl_paths(x1, x2, r_base=1.15, r_scale=0.55, ang_scale=1.05, cross=0.42):
    """
    Map (x1, x2) to two trajectories on the bowl: angle tracks self + partner,
    radius tracks partner (cross-coupling).
    """
    phi1 = ang_scale * x1 + cross * np.tanh(x2)
    phi2 = ang_scale * x2 + cross * np.tanh(x1) + 2 * np.pi / 3
    r1 = r_base + r_scale * np.tanh(x2)
    r2 = r_base + r_scale * np.tanh(x1)
    xa = r1 * np.cos(phi1)
    ya = r1 * np.sin(phi1)
    xb = r2 * np.cos(phi2)
    yb = r2 * np.sin(phi2)
    return xa, ya, xb, yb


def plot_bowl_two_balls_trajectory(t, x1, x2, *, r_max=2.8):
    """Static 3D bowl + full trajectories; plan view shows co-rotation."""
    X, Y, Z, k_parabola = bowl_surface_mesh(r_max=r_max)
    xa, ya, xb, yb = coupled_bowl_paths(x1, x2)
    za = k_parabola * (xa**2 + ya**2)
    zb = k_parabola * (xb**2 + yb**2)

    fig = plt.figure(figsize=(11, 5))
    ax3d = fig.add_subplot(1, 2, 1, projection="3d")
    ax3d.plot_surface(X, Y, Z, color="0.85", edgecolor="none", alpha=0.55, rstride=2, cstride=2)
    ax3d.plot(xa, ya, za, color="C0", lw=1.2, alpha=0.85, label="A trajectory")
    ax3d.plot(xb, yb, zb, color="C1", lw=1.2, alpha=0.85, label="B trajectory")
    ax3d.scatter([xa[-1]], [ya[-1]], [za[-1]], color="C0", s=80, depthshade=True)
    ax3d.scatter([xb[-1]], [yb[-1]], [zb[-1]], color="C1", s=80, depthshade=True)
    ax3d.set_title("Coupled marbles on a parabolic bowl")
    ax3d.set_xlabel("x")
    ax3d.set_ylabel("y")
    ax3d.set_zlabel("z")
    ax3d.legend(loc="upper left", fontsize=8)

    ax_top = fig.add_subplot(1, 2, 2)
    th = np.linspace(0, 2 * np.pi, 200)
    ax_top.plot(r_max * np.cos(th), r_max * np.sin(th), "k--", alpha=0.25, lw=1)
    ax_top.plot(xa, ya, color="C0", lw=1.0, alpha=0.8, label="A (plan)")
    ax_top.plot(xb, yb, color="C1", lw=1.0, alpha=0.8, label="B (plan)")
    ax_top.scatter([xa[-1]], [ya[-1]], color="C0", s=45, zorder=5)
    ax_top.scatter([xb[-1]], [yb[-1]], color="C1", s=45, zorder=5)
    ax_top.set_aspect("equal")
    ax_top.set_title("Plan view: orbits in the bowl")
    ax_top.set_xlabel("x")
    ax_top.set_ylabel("y")
    ax_top.legend(fontsize=8)
    ax_top.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def animate_bowl_two_balls(
    t,
    x1,
    x2,
    *,
    r_max=2.8,
    frame_step=None,
    interval_ms=35,
    rotate_view=True,
):
    """Animated marbles on the bowl; optional slow camera orbit."""
    X, Y, Z, k_parabola = bowl_surface_mesh(r_max=r_max)
    xa, ya, xb, yb = coupled_bowl_paths(x1, x2)
    za = k_parabola * (xa**2 + ya**2)
    zb = k_parabola * (xb**2 + yb**2)

    n = len(t)
    if frame_step is None:
        frame_step = max(1, n // 400)
    frames = np.arange(0, n, frame_step)

    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(projection="3d")
    ax.plot_surface(X, Y, Z, color="0.88", edgecolor="none", alpha=0.45, rstride=2, cstride=2)
    trail_a, = ax.plot([], [], [], color="C0", lw=1.0, alpha=0.65)
    trail_b, = ax.plot([], [], [], color="C1", lw=1.0, alpha=0.65)
    (ball_a,) = ax.plot([], [], [], "o", color="C0", markersize=10, label="Person A")
    (ball_b,) = ax.plot([], [], [], "o", color="C1", markersize=10, label="Person B")
    ax.set_xlim(-r_max, r_max)
    ax.set_ylim(-r_max, r_max)
    z_hi = float(np.nanmax(Z)) * 1.05
    ax.set_zlim(0, max(z_hi, 0.5))
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_title("Two coupled states as marbles in a bowl")
    ax.legend(loc="upper left", fontsize=9)

    def update(jj):
        j = int(frames[jj])
        trail_a.set_data(xa[: j + 1], ya[: j + 1])
        trail_a.set_3d_properties(za[: j + 1])
        trail_b.set_data(xb[: j + 1], yb[: j + 1])
        trail_b.set_3d_properties(zb[: j + 1])
        ball_a.set_data([xa[j]], [ya[j]])
        ball_a.set_3d_properties([za[j]])
        ball_b.set_data([xb[j]], [yb[j]])
        ball_b.set_3d_properties([zb[j]])
        if rotate_view:
            ax.view_init(elev=22, azim=45 + 0.35 * jj)
        return trail_a, trail_b, ball_a, ball_b

    anim = mpl_animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=interval_ms,
        blit=False,
    )
    plt.close(fig)
    try:
        from IPython.display import HTML

        return HTML(anim.to_jshtml())
    except Exception:
        return anim

# For static bowl plots after your own simulate():
# plot_bowl_two_balls_trajectory(t, x1, x2)
# animate_bowl_two_balls(t, x1, x2)

# Sliders + live bowl: run interactive_dashboard() in the next code cell.


## Interactive slider visualization

Run the **Two balls in a bowl** code cell above first so `bowl_surface_mesh`, `coupled_bowl_paths`, and `mpl_animation` exist.
Run the next cell in a local Jupyter environment. The `%matplotlib widget` backend needs the **ipympl** package (already listed in this project’s `pyproject.toml`). Install/sync the environment with `uv sync` if you see “widget is not a recognised backend”.

Use only one interactive backend per session; do not chain `%matplotlib notebook` after `widget`.

If the widget backend is unavailable, try `%matplotlib notebook` (classic nbagg) instead, or rely on static plots.

Running `interactive_dashboard()` opens **two figures**: the slider dashboard and a **bowl** window (3D + plan). Move sliders to re-simulate and restart the bowl animation.


In [ ]:
%matplotlib widget

def interactive_dashboard():
    """Sliders for all tunable params + second figure with bowl 3D/plan animation (restarts on change)."""
    params = LoveSystemParams()

    t, x1, x2, r1, r2, u1, u2 = simulate(params)
    r_max = 2.8

    fig = plt.figure(figsize=(15, 10))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.33, wspace=0.28, hspace=0.35)

    ax_state = fig.add_subplot(2, 2, 1)
    ax_reward = fig.add_subplot(2, 2, 2)
    ax_control = fig.add_subplot(2, 2, 3)
    ax_phase = fig.add_subplot(2, 2, 4)

    line_x1, = ax_state.plot(t, x1, label="x1: Person A")
    line_x2, = ax_state.plot(t, x2, label="x2: Person B")
    line_x1s = ax_state.axhline(params.x1_star, linestyle="--", alpha=0.7, label="A setpoint")
    line_x2s = ax_state.axhline(params.x2_star, linestyle=":", alpha=0.7, label="B setpoint")
    title_state = ax_state.set_title(f"Emotional States ({classify_regime(x1, x2)})")
    ax_state.set_xlabel("Time")
    ax_state.set_ylabel("State")
    ax_state.grid(True, alpha=0.3)
    ax_state.legend()

    line_r1, = ax_reward.plot(t, r1, label="r1: Reward felt by A")
    line_r2, = ax_reward.plot(t, r2, label="r2: Reward felt by B")
    ax_reward.set_title("Reward Signals")
    ax_reward.set_xlabel("Time")
    ax_reward.set_ylabel("Reward")
    ax_reward.grid(True, alpha=0.3)
    ax_reward.legend()

    line_u1, = ax_control.plot(t, u1, label="u1: A control")
    line_u2, = ax_control.plot(t, u2, label="u2: B control")
    ax_control.set_title("Setpoint Tracking Control")
    ax_control.set_xlabel("Time")
    ax_control.set_ylabel("Control input")
    ax_control.grid(True, alpha=0.3)
    ax_control.legend()

    line_phase, = ax_phase.plot(x1, x2, lw=1.2)
    start_pt = ax_phase.scatter([x1[0]], [x2[0]], s=60, marker="o", label="start")
    end_pt = ax_phase.scatter([x1[-1]], [x2[-1]], s=80, marker="x", label="end")
    ax_phase.set_title("Phase Portrait")
    ax_phase.set_xlabel("x1: Person A")
    ax_phase.set_ylabel("x2: Person B")
    ax_phase.grid(True, alpha=0.3)
    ax_phase.legend()

    Xb, Yb, Zb, k_parabola = bowl_surface_mesh(r_max=r_max)
    fig_bowl = plt.figure(figsize=(12, 5.2))
    fig_bowl.subplots_adjust(left=0.05, right=0.98, top=0.90, bottom=0.10, wspace=0.28)

    ax_b3d = fig_bowl.add_subplot(1, 2, 1, projection="3d")
    ax_b3d.plot_surface(Xb, Yb, Zb, color="0.88", edgecolor="none", alpha=0.45, rstride=2, cstride=2)
    bowl_trail_a, = ax_b3d.plot([], [], [], color="C0", lw=1.1, alpha=0.75, label="A")
    bowl_trail_b, = ax_b3d.plot([], [], [], color="C1", lw=1.1, alpha=0.75, label="B")
    (bowl_ball_a,) = ax_b3d.plot([], [], [], "o", color="C0", markersize=9, label="Person A")
    (bowl_ball_b,) = ax_b3d.plot([], [], [], "o", color="C1", markersize=9, label="Person B")
    ax_b3d.set_xlim(-r_max, r_max)
    ax_b3d.set_ylim(-r_max, r_max)
    z_hi = float(np.nanmax(Zb)) * 1.05
    ax_b3d.set_zlim(0, max(z_hi, 0.5))
    ax_b3d.set_title("Bowl — marbles (updates when you move sliders)")
    ax_b3d.set_xlabel("x")
    ax_b3d.set_ylabel("y")
    ax_b3d.set_zlabel("z")
    ax_b3d.legend(loc="upper left", fontsize=8)

    ax_bplan = fig_bowl.add_subplot(1, 2, 2)
    th = np.linspace(0, 2 * np.pi, 200)
    ax_bplan.plot(r_max * np.cos(th), r_max * np.sin(th), "k--", alpha=0.25, lw=1)
    plan_trail_a, = ax_bplan.plot([], [], color="C0", lw=1.0, alpha=0.85, label="A (plan)")
    plan_trail_b, = ax_bplan.plot([], [], color="C1", lw=1.0, alpha=0.85, label="B (plan)")
    plan_ball_a = ax_bplan.scatter([], [], color="C0", s=45, zorder=5)
    plan_ball_b = ax_bplan.scatter([], [], color="C1", s=45, zorder=5)
    ax_bplan.set_aspect("equal")
    ax_bplan.set_xlim(-r_max, r_max)
    ax_bplan.set_ylim(-r_max, r_max)
    ax_bplan.set_title("Plan view")
    ax_bplan.set_xlabel("x")
    ax_bplan.set_ylabel("y")
    ax_bplan.grid(True, alpha=0.3)
    ax_bplan.legend(fontsize=8)

    bowl_anim_holder = {"anim": None}

    def refresh_bowl(x1n, x2n, *, rotate_view=True):
        xa, ya, xb, yb = coupled_bowl_paths(x1n, x2n)
        za = k_parabola * (xa**2 + ya**2)
        zb = k_parabola * (xb**2 + yb**2)
        nlen = len(x1n)
        frame_step = max(1, nlen // 400)
        frames = np.arange(0, nlen, frame_step)

        old = bowl_anim_holder["anim"]
        if old is not None:
            try:
                old.event_source.stop()
            except Exception:
                pass
            bowl_anim_holder["anim"] = None

        def anim_update(jj):
            j = int(frames[jj])
            bowl_trail_a.set_data(xa[: j + 1], ya[: j + 1])
            bowl_trail_a.set_3d_properties(za[: j + 1])
            bowl_trail_b.set_data(xb[: j + 1], yb[: j + 1])
            bowl_trail_b.set_3d_properties(zb[: j + 1])
            bowl_ball_a.set_data([xa[j]], [ya[j]])
            bowl_ball_a.set_3d_properties([za[j]])
            bowl_ball_b.set_data([xb[j]], [yb[j]])
            bowl_ball_b.set_3d_properties([zb[j]])
            plan_trail_a.set_data(xa[: j + 1], ya[: j + 1])
            plan_trail_b.set_data(xb[: j + 1], yb[: j + 1])
            plan_ball_a.set_offsets(np.array([[xa[j], ya[j]]]))
            plan_ball_b.set_offsets(np.array([[xb[j], yb[j]]]))
            if rotate_view:
                ax_b3d.view_init(elev=22, azim=45 + 0.28 * jj)
            return (bowl_trail_a, bowl_trail_b, bowl_ball_a, bowl_ball_b)

        bowl_anim_holder["anim"] = mpl_animation.FuncAnimation(
            fig_bowl,
            anim_update,
            frames=len(frames),
            interval=32,
            blit=False,
            repeat=True,
        )
        fig_bowl.canvas.draw_idle()

    slider_specs = [
        ("a1", 0.0, 2.5, params.a1, [0.08, 0.26, 0.16, 0.02]),
        ("a2", 0.0, 2.5, params.a2, [0.30, 0.26, 0.16, 0.02]),
        ("b1", 0.0, 3.0, params.b1, [0.52, 0.26, 0.16, 0.02]),
        ("b2", 0.0, 3.0, params.b2, [0.74, 0.26, 0.16, 0.02]),
        ("k1", 0.0, 3.0, params.k1, [0.08, 0.22, 0.16, 0.02]),
        ("k2", 0.0, 3.0, params.k2, [0.30, 0.22, 0.16, 0.02]),
        ("K1", 0.0, 2.0, params.K1, [0.52, 0.22, 0.16, 0.02]),
        ("K2", 0.0, 2.0, params.K2, [0.74, 0.22, 0.16, 0.02]),
        ("tau1", 0.0, 8.0, params.tau1, [0.08, 0.18, 0.16, 0.02]),
        ("tau2", 0.0, 8.0, params.tau2, [0.30, 0.18, 0.16, 0.02]),
        ("sigma1", 0.0, 0.5, params.sigma1, [0.52, 0.18, 0.16, 0.02]),
        ("sigma2", 0.0, 0.5, params.sigma2, [0.74, 0.18, 0.16, 0.02]),
        ("x1*", -2.0, 2.0, params.x1_star, [0.08, 0.14, 0.16, 0.02]),
        ("x2*", -2.0, 2.0, params.x2_star, [0.30, 0.14, 0.16, 0.02]),
        ("sat", 0.1, 3.0, params.sat_alpha, [0.52, 0.14, 0.16, 0.02]),
        ("seed", 0, 100, params.seed, [0.74, 0.14, 0.16, 0.02]),
    ]

    sliders = {}
    for name, vmin, vmax, vinit, rect in slider_specs:
        sax = fig.add_axes(rect)
        valfmt = "%0.0f" if name == "seed" else "%1.2f"
        sliders[name] = Slider(ax=sax, label=name, valmin=vmin, valmax=vmax, valinit=vinit, valfmt=valfmt)

    reset_ax = fig.add_axes([0.82, 0.05, 0.12, 0.045])
    reset_button = Button(reset_ax, "Reset")

    def update(_=None):
        p = LoveSystemParams(
            T=params.T,
            dt=params.dt,
            x1_0=params.x1_0,
            x2_0=params.x2_0,
            shock_time_1=params.shock_time_1,
            shock_mag_1=params.shock_mag_1,
            shock_time_2=params.shock_time_2,
            shock_mag_2=params.shock_mag_2,
            a1=sliders["a1"].val,
            a2=sliders["a2"].val,
            b1=sliders["b1"].val,
            b2=sliders["b2"].val,
            k1=sliders["k1"].val,
            k2=sliders["k2"].val,
            K1=sliders["K1"].val,
            K2=sliders["K2"].val,
            tau1=sliders["tau1"].val,
            tau2=sliders["tau2"].val,
            sigma1=sliders["sigma1"].val,
            sigma2=sliders["sigma2"].val,
            x1_star=sliders["x1*"].val,
            x2_star=sliders["x2*"].val,
            sat_alpha=sliders["sat"].val,
            seed=int(sliders["seed"].val),
        )

        t2, x1n, x2n, r1n, r2n, u1n, u2n = simulate(p)

        line_x1.set_ydata(x1n)
        line_x2.set_ydata(x2n)
        line_r1.set_ydata(r1n)
        line_r2.set_ydata(r2n)
        line_u1.set_ydata(u1n)
        line_u2.set_ydata(u2n)
        line_phase.set_data(x1n, x2n)

        line_x1s.set_ydata([p.x1_star, p.x1_star])
        line_x2s.set_ydata([p.x2_star, p.x2_star])

        ax_state.relim()
        ax_state.autoscale_view()
        ax_reward.relim()
        ax_reward.autoscale_view()
        ax_control.relim()
        ax_control.autoscale_view()
        ax_phase.relim()
        ax_phase.autoscale_view()

        start_pt.set_offsets(np.array([[x1n[0], x2n[0]]]))
        end_pt.set_offsets(np.array([[x1n[-1], x2n[-1]]]))

        title_state.set_text(f"Emotional States ({classify_regime(x1n, x2n)})")
        fig.canvas.draw_idle()

        refresh_bowl(x1n, x2n, rotate_view=True)

    def reset(_):
        for s in sliders.values():
            s.eventson = False
        try:
            for s in sliders.values():
                s.reset()
        finally:
            for s in sliders.values():
                s.eventson = True
        update()

    for s in sliders.values():
        s.on_changed(update)

    reset_button.on_clicked(reset)

    refresh_bowl(x1, x2, rotate_view=True)
    plt.show()


interactive_dashboard()


## Suggested experiments

Try these regimes by moving the sliders:

### Stable
- `b1=1.0, b2=1.0`
- `k1=0.6, k2=0.6`
- `tau1=0.5, tau2=0.5`

### Oscillatory
- `b1=1.8, b2=1.7`
- `k1=1.0, k2=1.0`
- `tau1=2.5, tau2=2.5`

### One-sided
- `b1=1.8, b2=0.3`
- `k1=1.2, k2=0.2`

### Noisy
- `sigma1=0.15, sigma2=0.15`
- `tau1=2.0, tau2=3.0`